# FranchiseOps AI RAG Knowledge Base Builder
This notebook scrapes and curates a knowledge base for the FranchiseOps RAG system.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

!pip install -q langchain langchain-community langchain-text-splitters -U langchain-core sentence-transformers faiss-cpu pymupdf beautifulsoup4 requests==2.32.4 urllib3 tqdm vaderSentiment textblob

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
import json
import time
import requests
import fitz  # PyMuPDF
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import urllib3
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

RAG_DIR = '/content/drive/MyDrive/FranchiseOps_AI/rag_documents'
os.makedirs(RAG_DIR, exist_ok=True)
print(f'✅ Output Directory Ready: {RAG_DIR}')


In [ ]:
HTML_SOURCES = [
    # Marketing & Consumer Research
    "https://www.marketingweek.com",
    "https://hbr.org/topic/subject/marketing",
    "https://www.nielsen.com/insights",
    "https://www.mckinsey.com/capabilities/growth-marketing-and-sales/our-insights",
    "https://www.thinkwithgoogle.com",
    "https://www.campaignlive.co.uk",
    "https://www.warc.com/newsandopinion/opinion",

    # Customer Experience
    "https://www.pwc.com/us/en/services/consulting/library/consumer-intelligence-series.html",
    "https://www.zendesk.com/blog/customer-experience",
    "https://www.salesforce.com/resources/articles/customer-experience",

    # HR & Attrition Research
    "https://www.shrm.org/topics-tools/tools/how-to-guides/how-to-conduct-stay-interviews",
    "https://www.gallup.com/workplace/247391/fixable-problem-costs-businesses-trillion.aspx",
    "https://hbr.org/topic/subject/hr-management",
    "https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights",

    # Food Safety & FSSAI
    "https://www.fssai.gov.in",
    "https://www.fssai.gov.in/cms/food-safety-and-standards-act-2006.php",
    "https://www.fssai.gov.in/cms/rules.php",
    "https://www.fssai.gov.in/cms/regulations.php",
    "https://www.fssai.gov.in/cms/gazette-notifications.php",
    "https://www.fssai.gov.in/cms/licensing.php",

    # Labour Laws
    "https://labour.gov.in/minimum-wages-act",
    "https://labour.gov.in/payment-of-wages-act",
    "https://labour.gov.in/maternity-benefit-act",
    "https://labour.gov.in/child-labour",
    "https://labour.gov.in/factories-act",
    "https://labour.gov.in/employees-provident-fund-organisation",
    "https://labour.gov.in/employees-state-insurance-corporation",
    "https://labour.gov.in/occupational-safety-and-health",
    "https://labour.gov.in/social-security",
    "https://labour.gov.in/industrial-relations",
    "https://labour.gov.in/bonus-act",
    "https://labour.gov.in/gratuity-act",

    # OSHA
    "https://www.osha.gov/workers",
    "https://www.osha.gov/employers",
    "https://www.osha.gov/laws-regs",
    "https://www.osha.gov/heat-exposure",
    "https://www.osha.gov/young-workers",
    "https://www.osha.gov/ergonomics",
    "https://www.osha.gov/personal-protective-equipment",

    # FDA
    "https://www.fda.gov/food/guidance-regulation-food-and-dietary-supplements",
    "https://www.fda.gov/food/buy-store-serve-safe-food",
    "https://www.fda.gov/food/new-era-smarter-food-safety",
    "https://www.fda.gov/food/food-labeling-nutrition",

    # WHO & International
    "https://www.who.int/news-room/fact-sheets/detail/food-safety",
    "https://www.who.int/health-topics/food-safety",
    "https://www.codexalimentarius.org",
    "https://www.fao.org/food-safety/en",
    "https://efsa.europa.eu/en/topics/topic/food-safety",
    "https://www.epfindia.gov.in",
    "https://www.esic.gov.in",
    "https://www.bis.gov.in",
    "https://apeda.gov.in",
    "https://www.mofpi.gov.in",
    "https://niti.gov.in",
    "https://www.startupindia.gov.in",
    "https://www.msme.gov.in",
    "https://mca.gov.in",
    "https://consumerhelpline.gov.in",
    "https://www.ncdrc.nic.in",
    "https://www.ilo.org/global/topics/safety-and-health-at-work/lang--en/index.htm",
    "https://www.ilo.org/global/topics/wages/minimum-wages/lang--en/index.htm",
    "https://hbr.org/topic/subject/hr-management",
    "https://www.shrm.org/topics-tools/tools/how-to-guides/how-to-develop-employee-handbook",
    "https://www.foodsafety.gov",
    "https://www.food.gov.uk",
    "https://www.food.gov.uk/business-guidance",
    "https://www.foodstandards.gov.au",
    "https://www.canada.ca/en/health-canada/services/food-nutrition.html",
    "https://www.sqfi.com",
    "https://www.brcgs.com/our-standards/food-safety",
    "https://www.mygfsi.com",
    "https://www.iso.org/committee/47638.html",
    "https://www.fao.org/nutrition/en",
    "https://www.wcfc.co",
    "https://www.bfa.org.uk",
    "https://www.ftc.gov/tips-advice/business-center/guidance/franchise-rule",
    "https://www.sba.gov/business-guide/launch-your-business/buy-franchise",
    "https://www.ifa.com",
    "https://www.qsrmagazine.com",
    "https://www.nrn.com",
    "https://www.restaurant.org/research-and-media/research/research-reports/state-of-the-industry",
    "https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights",
    "https://www.gallup.com/workplace/247391/fixable-problem-costs-businesses-trillion.aspx",
    "https://www.nielsen.com/insights",
    "https://www.pwc.com/us/en/services/consulting/library/consumer-intelligence-series.html",
    "https://www.zendesk.com/blog/customer-experience",
    "https://www.salesforce.com/resources/articles/customer-experience",
    "https://www.indianspices.com",
    "https://www.coffeeboard.gov.in",
    "https://www.teaboard.gov.in",
    "https://mpeda.gov.in",
    "https://www.india.gov.in/spotlight/food-processing",
    "https://efsa.europa.eu/en/topics/topic/food-safety",
    "https://www.fao.org/food-safety/en",
    "https://www.who.int/teams/nutrition-and-food-safety/food-safety",
    "https://www.thinkwithgoogle.com",
    "https://www.marketingweek.com",
    "https://hbr.org/topic/subject/marketing"

    # -- Additional HTML sources (expansion batch) ------------------------
    "https://www.deloitte.com/global/en/Industries/consumer/perspectives.html",
    "https://www2.deloitte.com/us/en/insights/industry/retail-distribution.html",
    "https://www.pwc.com/gx/en/industries/consumer-markets.html",
    "https://www.ey.com/en_gl/consumer-products-retail",
    "https://home.kpmg/xx/en/home/industries/consumer-retail.html",
    "https://www.bcg.com/industries/consumer-products",
    "https://www.bain.com/industry-expertise/consumer-products/",
    "https://www.franchisedirect.com/information/franchisebusinessarticles",
    "https://www.entrepreneur.com/franchises",
    "https://www.franchising.com/articles",
    "https://www.qsrweb.com",
    "https://www.foodbusinessnews.net",
    "https://www.fooddive.com",
    "https://www.hrdive.com",
    "https://www.retaildive.com",
]

PDF_SOURCES = [
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Safety_and_Standards_Act_2006.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Licensing_and_Registration_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Packaging_and_Labelling_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Product_Standards_and_Food_Additives_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Prohibition_and_Restriction_on_Sales_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Contaminants_Toxins_and_Residues_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Laboratory_and_Sample_Analysis_Regulations_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Food_Safety_and_Standards_Rules_2011.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Compendium_Food_Safety_Standards_Act.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/44633/9789241501651_eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/255027/9789241512442-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/43038/9241546123_eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/42913/9241546468.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/326765/9789240004467-eng.pdf",
    "https://www.fao.org/3/a0512e/a0512e00.pdf",
    "https://www.fao.org/3/i3794e/i3794e.pdf",
    "https://www.fao.org/3/y1579e/y1579e00.pdf",
    "https://www.fao.org/3/w9raw-e.pdf",
    "https://www.fao.org/3/y1390e/y1390e00.pdf",
    "https://www.fao.org/3/a-i4955e.pdf",
    "https://www.fao.org/3/cb4474en/cb4474en.pdf",
    "https://www.fao.org/3/ca5399en/ca5399en.pdf",
    "https://www.fao.org/3/i0142e/i0142e.pdf",
    "https://www.fao.org/3/y4893e/y4893e.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_norm/---normes/documents/publication/wcms_087817.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---dgreports/---dcomm/documents/publication/wcms_067588.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---travail/documents/publication/wcms_712957.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_emp/---emp_ent/documents/publication/wcms_093580.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---asia/---ro-bangkok/---sro-new_delhi/documents/publication/wcms_631470.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3165.pdf",
    "https://www.osha.gov/sites/default/files/publications/3148-06R-2011-English.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3151.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha2254.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3170.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3071.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA_FS-3696.pdf",
    "https://www.ftc.gov/sites/default/files/documents/plain-language/bus70-franchise-rule.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/36613/9781464816109.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/35016/9781464816123.pdf",
    "https://labour.gov.in/sites/default/files/THE_MINIMUM_WAGES_ACT_1948.pdf",
    "https://labour.gov.in/sites/default/files/PaymentofWagesAct1936_0.pdf",
    "https://labour.gov.in/sites/default/files/TheMaternityBenefitAct_1961.pdf",
    "https://labour.gov.in/sites/default/files/payment_of_gratuity_act.pdf",
    "https://labour.gov.in/sites/default/files/ThePaymentofBonusAct1965.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXP+1-1969%2FCXP001e.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Organic_Foods.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Nutraceuticals.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Milk_and_Milk_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Meat_and_Meat_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Fruits_and_Vegetables.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Health_Supplements.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Food_Additives.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Document_Edible_Oils.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FSS_Organic_Foods_Regulations_2017.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Draft_FSS_Food_Recall_Procedure_Regulations_2017.pdf",
    "https://consumeronline.gov.in/documents/ConsumerProtection-Act-2019.pdf",
    "https://www.epfindia.gov.in/site_docs/PDFs/Circulars/Y2022-23/Circular_EPFO_0111_2022.pdf",
    "https://www.sba.gov/sites/default/files/2022-08/Franchise-Guide.pdf",
    "https://www.mckinsey.com/~/media/McKinsey/Business%20Functions/People%20and%20Organizational%20Performance/Our%20Insights/Reinventing%20the%20organization/Reinventing-the-organization.pdf",
    "https://www.foodstandards.gov.au/sites/default/files/documents/Meat%20Industry%20Operational%20Audit%20Guide.pdf",
    "https://www.ifi.unc.edu/wp-content/uploads/sites/863/2019/12/IFI-Report-Food-safety-culture.pdf",
    "https://efsa.onlinelibrary.wiley.com/doi/epdf/10.2903/j.efsa.2020.6098",
    "https://www.shrm.org/hr-today/trends-and-forecasting/research-and-surveys/Documents/SHRM%20Employee%20Job%20Satisfaction%20and%20Engagement.pdf",
    "https://www.gallup.com/workplace/349484/state-of-the-global-workplace-2022-report.aspx",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FOSTAC_Training_Module_Basic.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/FOSTAC_Training_Module_Special.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Hygiene_Rating_Scheme_Guidelines.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Clean_Street_Food_Hub_Guidelines.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---asia/---ro-bangkok/---sro-new_delhi/documents/publication/wcms_766448.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---safework/documents/instructionalmaterial/wcms_113522.pdf",
    "https://bis.gov.in/wp-content/uploads/2022/02/Annual-Report-2020-21.pdf",
    "https://apeda.gov.in/apedawebsite/ANNUAL_REPORT/APEDA-Annual-Report-2021-22.pdf",
    "https://www.msme.gov.in/sites/default/files/MSME-Annual-Report-2021-22.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/344474/9789240030060-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/274671/9789241514217-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/204348/9789241510066_eng.pdf",
    "https://www.fao.org/3/ca0640en/CA0640EN.pdf",
    "https://www.fao.org/3/i9933en/i9933en.pdf",
    "https://www.fao.org/3/cc0461en/cc0461en.pdf",
    "https://www.fao.org/3/cb7408en/cb7408en.pdf",
    "https://niti.gov.in/sites/default/files/2022-12/Food-Processing-Report.pdf",
    "https://www.mofpi.gov.in/sites/default/files/annual_report_2020-21.pdf",
    "https://www.restaurant.org/downloads/pdfs/research/whats_hot_2023.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA3604.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3180.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha_3590.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXS+1-1985%2FCXS001e.pdf",
    "https://www.fao.org/fao-who-codexalimentarius/sh-proxy/en/?lnk=1&url=https%3A%2F%2Fworkspace.fao.org%2Fsites%2Fcodex%2FStandards%2FCXS+193-1995%2FCXS193e.pdf"

    # -- Additional PDFs (expansion batch) --------------------------------
    "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Fortified_Foods.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Cereals_and_Cereal_Products.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Spices_and_Condiments.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Manual_Beverages_Confectionery_and_Chocolate.pdf",
    "https://www.fssai.gov.in/upload/uploadfiles/files/Eat_Right_India_Handbook.pdf",
    "https://labour.gov.in/sites/default/files/TheIndustrialDisputesAct1947.pdf",
    "https://labour.gov.in/sites/default/files/TheEmployeesCompensationAct1923.pdf",
    "https://labour.gov.in/sites/default/files/TheContractLabourAct1970.pdf",
    "https://labour.gov.in/sites/default/files/TheEqualRemunerationAct1976.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_dialogue/---lab_admin/documents/publication/wcms_120021.pdf",
    "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---ilo_aids/documents/publication/wcms_116048.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3990.pdf",
    "https://www.osha.gov/sites/default/files/publications/OSHA3708.pdf",
    "https://www.osha.gov/sites/default/files/publications/osha3111.pdf",
    "https://www.ftc.gov/system/files/documents/plain-language/bus70-buying-franchise-consumer-guide.pdf",
    "https://www.sba.gov/sites/default/files/2019-08/Small-Business-Trends.pdf",
    "https://www.irs.gov/pub/irs-pdf/p334.pdf",
    "https://www.irs.gov/pub/irs-pdf/p583.pdf",
    "https://www.irs.gov/pub/irs-pdf/p15.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/337288/9789240012456-eng.pdf",
    "https://apps.who.int/iris/bitstream/handle/10665/241016/9789241549752-eng.pdf",
    "https://www.fao.org/3/cb9479en/cb9479en.pdf",
    "https://www.fao.org/3/i8347en/i8347en.pdf",
    "https://www.fao.org/3/ca9692en/ca9692en.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/33547/9781464815614.pdf",
    "https://openknowledge.worldbank.org/bitstream/handle/10986/30436/9781464812159.pdf",
    "https://www.epfindia.gov.in/site_docs/PDFs/Downloads_PDFs/EmployeesPFScheme1952.pdf",
    "https://www.mofpi.gov.in/sites/default/files/pmfme_guidelines.pdf",
    "https://niti.gov.in/sites/default/files/2021-06/FoodProcessingSectorReport.pdf",
    "https://www.restaurant.org/downloads/pdfs/research/2022-state-of-the-industry-mid-year-update.pdf",
]

print(f"HTML Sources: {len(HTML_SOURCES)}")
print(f"PDF Sources: {len(PDF_SOURCES)}")

In [ ]:
import os, json, time, requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import fitz  # PyMuPDF
from tqdm import tqdm

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

manifest_path = os.path.join(RAG_DIR, 'manifest.json')
manifest = {}
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

def save_manifest():
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

def get_with_retry(url, max_retries=3, timeout=20):
    """GET request with exponential backoff and SSL fallback."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=True)
            resp.raise_for_status()
            return resp
        except requests.exceptions.SSLError:
            try:
                resp = requests.get(url, headers=HEADERS, timeout=timeout, verify=False)
                resp.raise_for_status()
                return resp
            except Exception as e:
                if attempt == max_retries - 1:
                    raise e
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            wait = 2 ** attempt
            time.sleep(wait)
    return None

def harvest_pdfs_from_page(url, base_domain=None):
    """
    Visit a webpage and auto-discover all PDF links embedded inside it.
    Returns a list of absolute PDF URLs found on the page.
    """
    discovered = []
    try:
        resp = get_with_retry(url, timeout=15)
        if resp is None:
            return discovered
        soup = BeautifulSoup(resp.content, 'html.parser')
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            # Check if link ends with .pdf or contains /pdf/ in path
            if href.lower().endswith('.pdf') or '/pdf/' in href.lower():
                # Convert relative URLs to absolute
                if href.startswith('http'):
                    abs_url = href
                elif href.startswith('//'):
                    abs_url = 'https:' + href
                elif href.startswith('/'):
                    parsed = urlparse(url)
                    abs_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                else:
                    abs_url = urljoin(url, href)
                # Clean URL (remove fragments)
                abs_url = abs_url.split('#')[0]
                if abs_url not in discovered:
                    discovered.append(abs_url)
    except Exception as e:
        print(f"  ⚠️  PDF harvest failed for {url}: {e}")
    return discovered

def scrape_html(url):
    """Scrape HTML page text and save as .txt file."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        soup = BeautifulSoup(resp.content, 'html.parser')
        # Remove script and style elements
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        if len(text.strip()) < 100:
            manifest[url] = 'failed: too short'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'html_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

def scrape_pdf(url):
    """Download a PDF and extract all text pages."""
    if manifest.get(url) == 'success':
        return 'skipped'
    try:
        resp = get_with_retry(url, timeout=45)
        if resp is None:
            manifest[url] = 'failed: no response'
            return 'failed'
        # Check content type
        content_type = resp.headers.get('Content-Type', '')
        if 'pdf' not in content_type.lower() and not url.lower().endswith('.pdf'):
            manifest[url] = 'failed: not a pdf'
            return 'failed'
        doc = fitz.open(stream=resp.content, filetype='pdf')
        text = '\n'.join([page.get_text() for page in doc])
        doc.close()
        if len(text.strip()) < 50:
            manifest[url] = 'failed: empty pdf (scanned image)'
            return 'failed'
        safe_name = url.replace('https://', '').replace('http://', '').replace('/', '_').replace('?', '_')[:80]
        out_path = os.path.join(RAG_DIR, f'pdf_{safe_name}.txt')
        with open(out_path, 'w', encoding='utf-8', errors='ignore') as f:
            f.write(f'SOURCE: {url}\n\n{text}')
        manifest[url] = 'success'
        save_manifest()
        return 'success'
    except Exception as e:
        manifest[url] = f'failed: {str(e)[:80]}'
        save_manifest()
        return 'failed'

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: Scrape all HTML pages AND auto-harvest PDF links from each page
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1: Scraping HTML pages + Auto-harvesting embedded PDFs')
print('=' * 60)

discovered_pdfs = set()
html_stats = {'success': 0, 'skipped': 0, 'failed': 0}

for url in tqdm(HTML_SOURCES, desc='🌐 HTML Pages'):
    result = scrape_html(url)
    html_stats[result] = html_stats.get(result, 0) + 1

    # Auto-harvest PDF links from every HTML page
    if result in ('success', 'skipped'):
        pdfs_found = harvest_pdfs_from_page(url)
        for pdf_url in pdfs_found:
            discovered_pdfs.add(pdf_url)
        if pdfs_found:
            print(f'  📎 Found {len(pdfs_found)} PDFs on {url[:60]}')
    time.sleep(1)  # Polite delay

print(f'\n✅ HTML done: {html_stats}')
print(f'📎 Auto-discovered {len(discovered_pdfs)} PDFs from HTML pages')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Merge discovered PDFs with static PDF_SOURCES list
# ─────────────────────────────────────────────────────────────────────────────
all_pdf_urls = list(set(PDF_SOURCES) | discovered_pdfs)
print(f'\n📚 Total unique PDFs to process: {len(all_pdf_urls)}')
print(f'  → From static list: {len(PDF_SOURCES)}')
print(f'  → Auto-discovered:  {len(discovered_pdfs)}')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: Download & extract all PDFs
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('PHASE 3: Downloading & Extracting All PDFs')
print('=' * 60)

pdf_stats = {'success': 0, 'skipped': 0, 'failed': 0}
for url in tqdm(all_pdf_urls, desc='📄 PDFs'):
    result = scrape_pdf(url)
    pdf_stats[result] = pdf_stats.get(result, 0) + 1
    if result == 'success':
        time.sleep(0.5)  # Polite delay for successful downloads

print(f'\n✅ PDF done: {pdf_stats}')

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
txt_files = [f for f in os.listdir(RAG_DIR) if f.endswith('.txt')]
print('\n' + '=' * 60)
print('📊 SCRAPING COMPLETE — SUMMARY')
print('=' * 60)
print(f'  HTML pages scraped:      {html_stats["success"]} success, {html_stats["skipped"]} skipped, {html_stats["failed"]} failed')
print(f'  PDFs auto-discovered:    {len(discovered_pdfs)}')
print(f'  PDFs downloaded:         {pdf_stats["success"]} success, {pdf_stats["skipped"]} skipped, {pdf_stats["failed"]} failed')
print(f'  Total .txt files in RAG: {len(txt_files)}')
print(f'  Manifest entries:        {len(manifest)}')
print('=' * 60)


In [ ]:
print(f"\nLoading {len(txt_files)} scraped text files from Drive...")
documents = []
for fname in tqdm(txt_files, desc="📂 Loading Docs"):
    filepath = os.path.join(RAG_DIR, fname)
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    if len(text.strip()) > 50:
        documents.append(Document(page_content=text, metadata={"source": fname, "type": "scraped"}))

print(f"\n✅ Loaded {len(documents)} documents into memory.")


In [ ]:
curated_sops = [
    {"id": "KB-101", "content": "Minimum freezer temperature must be maintained at -18°C or below at all times."},
    {"id": "KB-102", "content": "Store closing procedures include counting the cash drawer, securing the safe, turning off non-essential equipment, and setting the alarm."},
    {"id": "KB-103", "content": "Staff must wash hands with soap and warm water for at least 20 seconds before starting a shift, after using the restroom, and after handling raw meat."},
    {"id": "KB-104", "content": "Food preparation surfaces must be sanitized every 2 hours using the approved quaternary ammonium sanitizer solution."},
    {"id": "KB-105", "content": "The FIFO (First In, First Out) method must be strictly followed for all perishable inventory."},
    {"id": "KB-106", "content": "Daily temperature logs for all refrigeration units must be recorded at 8:00 AM, 2:00 PM, and 8:00 PM."},
    {"id": "KB-107", "content": "Customer complaints regarding food quality must be immediately escalated to the Shift Manager for resolution."},
    {"id": "KB-108", "content": "Spills on the customer floor must be marked with a wet floor sign and cleaned up within 3 minutes."},
    {"id": "KB-109", "content": "All staff members must wear the complete, approved uniform including name tag, hat/visor, and slip-resistant shoes."},
    {"id": "KB-110", "content": "Waste bins must be emptied when they are 3/4 full; never allow trash to overflow."},
    {"id": "KB-111", "content": "A minimum of 3 staff members (1 Manager, 1 Front-of-House, 1 Back-of-House) are required per shift."},
    {"id": "KB-112", "content": "FSSAI license must be prominently displayed near the point of sale at all times."},
    {"id": "KB-113", "content": "Penalties for FSSAI non-compliance can range from warning letters to license suspension and fines up to ₹2,00,000 depending on the severity."},
    {"id": "KB-114", "content": "Pest control services must be scheduled monthly, and inspection reports must be kept in the compliance binder."},
    {"id": "KB-115", "content": "Only approved vendors may be used for sourcing raw ingredients and packaging materials."},
    {"id": "KB-116", "content": "Deep fryers must be filtered daily and the oil completely changed every 3 days or when it fails the color check test."},
    {"id": "KB-117", "content": "Fire extinguishers must be inspected monthly by the Manager and annually by a certified professional."},
    {"id": "KB-118", "content": "All new employees must complete the 40-hour basic operational training program before working independently."},
    {"id": "KB-119", "content": "Cash drops to the safe must be performed whenever the register drawer exceeds ₹20,000."},
    {"id": "KB-120", "content": "The store key must never be duplicated, and must be returned immediately upon termination of employment."},
    {"id": "KB-121", "content": "Marketing ROI minimum threshold: >15% ROI is required for any marketing campaign renewal."},
    {"id": "KB-122", "content": "Customer complaint categories and resolution SLA per category: Critical (2 hours), High (24 hours), Medium (48 hours), Low (72 hours)."},
    {"id": "KB-123", "content": "Staff performance review frequency is quarterly, utilizing a 5-point scoring methodology based on attendance, customer service, and task completion."},
    {"id": "KB-124", "content": "Social media compliance requires a response within 2 hours for all negative reviews on major platforms."},
    {"id": "KB-125", "content": "Outlet opening checklist includes a 30-point pre-launch audit covering equipment, inventory, staff readiness, and regulatory compliance."},
    {"id": "KB-126", "content": "HACCP critical control points must be documented and reviewed weekly by the Food Safety Officer."},
    {"id": "KB-127", "content": "Allergen labeling must clearly identify all 14 major allergens on packaged items sold at the outlet."},
    {"id": "KB-128", "content": "Cross-contamination prevention requires separate cutting boards color-coded for raw meat (red), poultry (yellow), seafood (blue), and vegetables (green)."},
    {"id": "KB-129", "content": "Hot-held food must be maintained at or above 63 degrees Celsius (145F) and checked every 2 hours with a calibrated thermometer."},
    {"id": "KB-130", "content": "Cold-held food must be maintained at or below 5 degrees Celsius (41F) at all times during service."},
    {"id": "KB-131", "content": "Any food item left in the temperature danger zone (5C-63C) for more than 4 hours must be discarded immediately."},
    {"id": "KB-132", "content": "Hand sanitizer stations must be refilled daily and placed at all customer-facing entry points."},
    {"id": "KB-133", "content": "Expired inventory must be logged in the waste tracking sheet with reason code and disposed of per local health regulations."},
    {"id": "KB-134", "content": "Water used for food preparation must be tested for potability quarterly and results filed in the compliance binder."},
    {"id": "KB-135", "content": "All food handlers must hold a valid Food Safety Supervisor certificate renewed every 3 years."},
    {"id": "KB-136", "content": "New hire onboarding must be completed within the first 3 working days, including safety training and POS system walkthrough."},
    {"id": "KB-137", "content": "Overtime pay is calculated at 1.5x the base hourly rate for any hours worked beyond 48 hours per week."},
    {"id": "KB-138", "content": "Employees are entitled to a minimum 30-minute unpaid break for every 6 hours worked."},
    {"id": "KB-139", "content": "Grievance redressal requests must be acknowledged by HR within 48 hours and resolved within 15 working days."},
    {"id": "KB-140", "content": "Exit interviews are mandatory for all voluntary resignations and must be conducted by a manager not directly supervising the employee."},
    {"id": "KB-141", "content": "Maternity leave entitlement is 26 weeks of paid leave as per the Maternity Benefit Act."},
    {"id": "KB-142", "content": "Employee referral bonus is Rs 5,000 payable after the referred candidate completes 90 days of employment."},
    {"id": "KB-143", "content": "Disciplinary action follows a 3-step process: verbal warning, written warning, and final termination review."},
    {"id": "KB-144", "content": "Annual leave accrual is 1.5 days per completed month of service, capped at 18 days per year."},
    {"id": "KB-145", "content": "Employee Provident Fund (EPF) contributions must be deposited with EPFO by the 15th of every month."},
    {"id": "KB-146", "content": "Net Promoter Score (NPS) surveys must be sent to customers within 24 hours of their visit."},
    {"id": "KB-147", "content": "Loyalty program points expire after 12 months of account inactivity."},
    {"id": "KB-148", "content": "In-store promotional signage must be updated within 48 hours of a campaign launch or expiry."},
    {"id": "KB-149", "content": "Customer refunds for quality issues must be processed within the same business day without requiring managerial escalation for amounts under Rs 500."},
    {"id": "KB-150", "content": "Social media response time target for direct messages is under 1 hour during business hours."},
    {"id": "KB-151", "content": "Local store marketing budget allocation is capped at 3% of monthly gross revenue."},
    {"id": "KB-152", "content": "Customer satisfaction score (CSAT) below 3.5 out of 5 for two consecutive months triggers a mandatory service quality audit."},
    {"id": "KB-153", "content": "Franchisees must use only brand-approved marketing creative assets; unauthorized local advertising requires corporate sign-off."},
    {"id": "KB-154", "content": "Franchise agreement renewal notice must be submitted to the franchisor at least 180 days before contract expiration."},
    {"id": "KB-155", "content": "Royalty payments are due to the franchisor by the 5th business day of each month, calculated at the contracted percentage of gross sales."},
    {"id": "KB-156", "content": "Territory exclusivity radius is defined in the franchise agreement and typically spans 3-5 kilometers depending on market density."},
    {"id": "KB-157", "content": "Any change in outlet ownership or majority shareholding must be disclosed to the franchisor within 30 days."},
    {"id": "KB-158", "content": "Non-compete clauses restrict former franchisees from operating a competing business within the exclusive territory for 24 months post-termination."},
    {"id": "KB-159", "content": "Annual franchise disclosure document (FDD) updates must be reviewed and signed by the franchisee before the fiscal year renewal."},
    {"id": "KB-160", "content": "Trademark and brand usage must strictly follow the Brand Standards Manual; unauthorized logo modifications are a material breach."},
    {"id": "KB-161", "content": "Insurance coverage minimums include general liability, workers compensation, and property damage as specified in Schedule C of the franchise agreement."},
    {"id": "KB-162", "content": "Daily cash reconciliation must balance within a Rs 100 variance threshold; discrepancies beyond this require a written incident report."},
    {"id": "KB-163", "content": "Petty cash fund is capped at Rs 10,000 and must be replenished only against original receipts."},
    {"id": "KB-164", "content": "Monthly P&L statements must be submitted to the corporate finance team by the 7th of the following month."},
    {"id": "KB-165", "content": "Vendor payment terms are net-30 unless otherwise negotiated and documented in the vendor contract."},
    {"id": "KB-166", "content": "Capital expenditure requests above Rs 1,00,000 require regional manager approval before procurement."},
    {"id": "KB-167", "content": "Sales tax and GST filings must be reconciled and submitted quarterly in accordance with statutory deadlines."},
    {"id": "KB-168", "content": "POS system downtime must be reported to IT support within 15 minutes, with manual transaction logging as the fallback procedure."},
    {"id": "KB-169", "content": "Customer data collected through loyalty programs must be stored in compliance with applicable data protection regulations and never shared with third parties without consent."},
    {"id": "KB-170", "content": "Software and firmware updates to the POS terminals must be scheduled during non-peak hours, typically between 2 AM and 4 AM."},
    {"id": "KB-171", "content": "CCTV footage must be retained for a minimum of 90 days and reviewed immediately in the event of a reported incident."},
    {"id": "KB-172", "content": "Single-use plastic packaging must be phased out in favor of biodegradable alternatives per the corporate sustainability roadmap."},
    {"id": "KB-173", "content": "Cooking oil waste must be collected by a licensed recycler and never disposed of through regular drainage systems."},
    {"id": "KB-174", "content": "Energy audits are conducted annually to identify opportunities for reducing outlet electricity consumption."},
    {"id": "KB-175", "content": "Food waste diversion programs must track and report the percentage of waste composted or donated versus landfilled each quarter."}
]

with open('kb_franchise.json', 'w') as f:
    json.dump(curated_sops, f, indent=4)

for sop in curated_sops:
    documents.append(Document(page_content=sop["content"], metadata={"source": "curated_sop", "id": sop["id"]}))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("franchiseops_faiss_index")
print("FAISS index built and saved successfully.")

In [ ]:
print("\n--- ROBUSTNESS TEST: 32 Distinct Queries ---")

# Each entry: (query, expected_keyword_that_must_appear_in_the_retrieved_chunk)
# Every keyword below has been verified to appear in EXACTLY ONE curated_sops
# entry (see verification script), so a PASS here means the correct chunk was
# retrieved -- not just that some coincidental substring matched.
test_queries = [
    ("What is the minimum freezer temperature?", "-18"),
    ("How many staff are required per shift?", "3 staff members"),
    ("What is the handwashing procedure?", "20 seconds"),
    ("What are the penalties for FSSAI non-compliance?", "2,00,000"),
    ("How should customer complaints be escalated?", "Shift Manager"),
    ("What is the minimum marketing ROI threshold for campaign renewal?", "15% ROI"),
    ("What are the staff performance review requirements?", "5-point scoring"),
    ("What are the critical control points in HACCP?", "Food Safety Officer"),
    ("How many major allergens must be labeled?", "14 major allergens"),
    ("What color cutting board is used for raw meat?", "color-coded for raw meat"),
    ("What temperature must hot-held food be maintained at?", "calibrated thermometer"),
    ("What temperature must cold-held food be kept at?", "5 degrees Celsius"),
    ("How long can food stay in the temperature danger zone?", "discarded immediately"),
    ("How often must potable water be tested?", "tested for potability"),
    ("How long is a Food Safety Supervisor certificate valid?", "renewed every 3 years"),
    ("How soon must new hire onboarding be completed?", "3 working days"),
    ("What is the overtime pay rate?", "1.5x the base hourly rate"),
    ("How long is the mandatory break for a 6-hour shift?", "30-minute unpaid break"),
    ("How quickly must HR acknowledge a grievance?", "acknowledged by HR within 48 hours"),
    ("What is the maternity leave entitlement?", "26 weeks"),
    ("What is the employee referral bonus amount?", "referral bonus is Rs 5,000"),
    ("What is the annual leave accrual rate?", "1.5 days per completed month"),
    ("When must EPF contributions be deposited?", "15th of every month"),
    ("How soon must NPS surveys be sent after a visit?", "surveys must be sent"),
    ("When do loyalty points expire?", "expire after 12 months"),
    ("What is the local marketing budget cap?", "capped at 3%"),
    ("What CSAT score triggers a service quality audit?", "3.5 out of 5"),
    ("How far in advance must a franchise renewal notice be submitted?", "180 days"),
    ("When are royalty payments due to the franchisor?", "5th business day"),
    ("What is the non-compete period after franchise termination?", "24 months post-termination"),
    ("What is the daily cash reconciliation variance threshold?", "Rs 100 variance"),
    ("How long must CCTV footage be retained?", "CCTV footage must be retained")
]

assert len(test_queries) >= 30, f"Need 30+ test queries, found {len(test_queries)}"

passed, failed = 0, 0
for query, expected_keyword in test_queries:
    docs = vectorstore.similarity_search(query, k=1)
    if docs and expected_keyword.lower() in docs[0].page_content.lower():
        status = "PASS"
        passed += 1
    else:
        status = "FAIL"
        failed += 1
    src_id = docs[0].metadata.get("id", docs[0].metadata.get("source", "Unknown")) if docs else "N/A"
    print(f"[{status}] {query}  -> matched: {src_id}")

print("\n" + "=" * 60)
print(f"ROBUSTNESS TEST SUMMARY: {passed}/{len(test_queries)} PASSED, {failed}/{len(test_queries)} FAILED")
print("=" * 60)
if failed == 0:
    print("All queries passed. RAG pipeline verified robust across", len(test_queries), "distinct queries.")
else:
    print("Review FAILED queries above -- check chunk_size/overlap or rephrase the query.")
